# `LLMToolEmulator`

Middleware that emulates tool execution using a language model instead of running the real tool.

It is intended mainly for testing agents without calling external APIs, changing files, accessing databases, or performing other real tool-side effects.

By default, every tool is emulated. A selected set of tools can be emulated by passing tool names or `BaseTool` instances.

- Bases: `AgentMiddleware[AgentState[Any], ContextT]`
- Generic over: `ContextT`

## Constructor

```python
LLMToolEmulator(
    *,
    tools: list[str | BaseTool] | None = None,
    model: str | BaseChatModel | None = None
)
```

## Parameters

* `tools` — Tools that should be emulated.
  * Default: `None`
  * `None` — Emulates every tool call.
  * Empty list `[]` — Emulates no tools.
  * List of strings — Emulates tools having those names.
  * List of `BaseTool` objects — Emulates tools using their `.name` values.
  * A list may contain both strings and `BaseTool` instances.

* `model` — Model used to generate simulated tool results.
  * Default: `"anthropic:claude-sonnet-4-5-20250929"`
  * May be a model identifier string.
  * May be an initialized `BaseChatModel`.
  * String model identifiers are initialized with `temperature=1`.
  * The default model is also initialized with `temperature=1`.

## Attributes

* `emulate_all` — Indicates whether every tool should be emulated.
  * Type: `bool`
  * Set to `True` only when `tools=None`.

* `tools_to_emulate` — Set containing the names of selectively emulated tools.
  * Type: `set[str]`
  * Empty when all tools are emulated.
  * Also empty when `tools=[]`, but `emulate_all` distinguishes these two cases.

* `model` — Initialized chat model used to generate simulated tool output.
  * Type: `BaseChatModel`

## Tool Selection Behaviour

The constructor interprets `tools` as follows:

| Configuration | Behaviour |
|---|---|
| `tools=None` | Emulate all tools |
| `tools=[]` | Emulate no tools |
| `tools=["search"]` | Emulate only `search` |
| `tools=[search_tool]` | Emulate the tool named by `search_tool.name` |
| `tools=["search", calculator_tool]` | Emulate both selected names |

The decision used for every tool call is:

```python
should_emulate = (
    self.emulate_all
    or tool_name in self.tools_to_emulate
)
```

When `should_emulate` is `False`, the middleware calls the normal tool handler.

When `should_emulate` is `True`, the real tool is not executed.

## Methods

### 1. `wrap_tool_call`

Intercepts a synchronous tool call.

If the selected tool should be emulated, the method builds a prompt describing the tool and its arguments, sends that prompt to the emulator model, and returns the generated response as a `ToolMessage`.

If the tool should not be emulated, it delegates to the supplied handler for normal execution.

```python
wrap_tool_call(
    self,
    request: ToolCallRequest,
    handler: Callable[
        [ToolCallRequest],
        ToolMessage | Command[Any]
    ]
) -> ToolMessage | Command[Any]
```

#### Parameters

* `request` — Tool-call request containing:
  * The tool-call name.
  * Tool arguments.
  * Tool-call ID.
  * The bound tool object, when available.

* `handler` — Callback that executes the actual tool.
  * Called only when the tool is not selected for emulation.

#### Returns

* An emulated `ToolMessage` when the tool is selected.
* The handler's `ToolMessage` or `Command` result when normal execution is allowed.

### 2. `awrap_tool_call`

Asynchronous version of `wrap_tool_call`.

It applies the same selection and prompt-building logic, but uses `model.ainvoke` and awaits the normal tool handler when the tool is not emulated.

```python
async def awrap_tool_call(
    self,
    request: ToolCallRequest,
    handler: Callable[
        [ToolCallRequest],
        Awaitable[
            ToolMessage | Command[Any]
        ]
    ]
) -> ToolMessage | Command[Any]
```

## Emulation Prompt

For an emulated tool call, the middleware creates a prompt containing:

* The tool name.
* The tool description.
* The supplied arguments.
* Instructions to generate realistic output.
* Instructions to return only the tool result.
* Instructions to introduce variation.

Its structure is equivalent to:

```text
You are emulating a tool call for testing purposes.

Tool: <tool name>
Description: <tool description>
Arguments: <tool arguments>

Generate a realistic response that this tool would return
given these arguments.
Return ONLY the tool's output, no explanation or preamble.
Introduce variation into your responses.
```

When the request does not contain a bound tool object, the description becomes:

```text
No description available
```

## Emulated Result

The model response is converted into:

```python
ToolMessage(
    content=response.content,
    tool_call_id=request.tool_call["id"],
    name=tool_name,
)
```

The generated result:

* Uses the original tool-call ID.
* Uses the original tool name.
* Does not execute the real tool.
* Does not include an explicit success or error status.
* Does not return a `Command` or update agent state.

## Execution Flow

```text
Receive ToolCallRequest
        |
        v
Read request.tool_call["name"]
        |
        v
Should this tool be emulated?
        |
   No --+--> Call the real tool handler
        |
       Yes
        |
        v
Read tool arguments and description
        |
        v
Build the emulator prompt
        |
        v
Invoke the emulator model
        |
        v
Return generated ToolMessage
```

## Synchronous Flow

```python
response = self.model.invoke(
    [HumanMessage(prompt)]
)
```

## Asynchronous Flow

```python
response = await self.model.ainvoke(
    [HumanMessage(prompt)]
)
```

## Examples

### Emulate All Tools

```python
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolEmulator

middleware = LLMToolEmulator()

agent = create_agent(
    model="openai:gpt-5.5",
    tools=[
        get_weather,
        get_user_location,
        calculator,
    ],
    middleware=[middleware],
)
```

Because `tools=None`, none of the real tools are executed.

### Emulate Selected Tools by Name

```python
middleware = LLMToolEmulator(
    tools=[
        "get_weather",
        "get_user_location",
    ]
)
```

Calls to those two names are emulated. Other tools execute normally.

### Emulate Selected Tool Instances

```python
middleware = LLMToolEmulator(
    tools=[
        get_weather,
        get_user_location,
    ]
)
```

The middleware extracts:

```python
get_weather.name
get_user_location.name
```

and stores them in `tools_to_emulate`.

### Use a Custom Emulator Model

```python
middleware = LLMToolEmulator(
    tools=["get_weather"],
    model="anthropic:claude-sonnet-4-5-20250929",
)
```

### Use an Initialized Model

```python
from langchain_openai import ChatOpenAI

emulator_model = ChatOpenAI(
    model="gpt-5.5",
    temperature=1,
)

middleware = LLMToolEmulator(
    tools=["get_weather"],
    model=emulator_model,
)
```

When an initialized `BaseChatModel` is supplied, the middleware uses it directly and does not override its configuration.

### Disable Emulation

```python
middleware = LLMToolEmulator(
    tools=[]
)
```

Every tool call is passed to its normal handler.

### Mix Emulated and Real Tools

```python
middleware = LLMToolEmulator(
    tools=["get_weather"]
)

agent = create_agent(
    model="openai:gpt-5.5",
    tools=[
        get_weather,
        calculator,
    ],
    middleware=[middleware],
)
```

Behaviour:

```text
get_weather -> LLM-generated result
calculator  -> Real tool execution
```

## Important Behaviour

### No Tool-Name Validation

The constructor does not verify that the names in `tools` are actually bound to the agent.

For example:

```python
LLMToolEmulator(
    tools=["unknown_tool"]
)
```

does not raise an error during initialization. It simply emulates a call only if a tool call later uses that exact name.

### No Real Side Effects

An emulated tool does not:

* Call an external API.
* Read from or write to the real tool's backend.
* Modify files through the real tool.
* Execute database operations.
* Produce state updates returned by the actual tool.
* Run custom validation or business logic inside the real tool.

Only an LLM-generated `ToolMessage` is returned.

### Generated Results Are Not Guaranteed to Be Accurate

The emulator model is instructed to create a realistic result, but the output is synthetic.

It may be:

* Factually incorrect.
* Inconsistent with a real backend.
* Different across repeated calls.
* Incompatible with strict downstream output expectations.

The default use of `temperature=1` intentionally encourages variation.

### Tool Description Affects Quality

When `request.tool` is available, its `.description` is included in the prompt.

Clear tool descriptions generally produce more realistic emulated results.

When no tool object is available, the model receives only:

```text
No description available
```

### Return Type Difference

A real tool handler may return:

```python
ToolMessage | Command[Any]
```

An emulated call always returns:

```python
ToolMessage
```

Therefore, emulation does not reproduce `Command`-based state changes made by the real tool.

## Typical Use Cases

`LLMToolEmulator` is useful for:

* Agent integration tests.
* UI demonstrations.
* Development without external credentials.
* Testing tool-selection logic.
* Simulating unavailable APIs.
* Avoiding destructive side effects during evaluation.

It is less suitable when tests must verify:

* Exact tool output.
* Real external-system behaviour.
* Tool-side validation.
* Database or file changes.
* `Command`-based state updates.
* Deterministic repeatability.

## Exceptions

The middleware does not define a custom exception type.

Exceptions may still propagate from:

* `init_chat_model`.
* The emulator model's `invoke` or `ainvoke`.
* The normal tool handler for non-emulated tools.
* Missing or malformed required fields in `request.tool_call`.

## Source

This reference follows the pinned LangChain source:

```text
libs/langchain_v1/langchain/agents/middleware/tool_emulator.py
Commit: 42f8f79293cfb7589e5bc1d74a8ae4dfd0bf15e3
```